In [ ]:
# Installe la bibliothèque pour le rendu du labyrinthe
!pip install pygame

In [ ]:
# Importation des modules nécessaires
import pygame,sys
from pygame.draw import *
from pygame.display import *
from pygame.locals import * 
from random import randint

In [ ]:
# Gestion des noeuds définis par un numéro, des coordonnées, un poids et deux booléens
class Node :
    
    def __init__(self,value,weight) :
        self.value = value
        self.coords = (0,0)   # Coordonnées par défaut
        self.weight = weight  # Poids de l'arête la reliant à son parent
        self.on_use = False   # Noeud en cours d'utilisation en tant que futur parent possible
        self.is_used = False  # Noeud déjà utilisé en tant qu'enfant

        
##################################################################################################
##################################################################################################
##################################################################################################

In [ ]:
# Gestion du graphe
class Graph :
    
    def __init__(self, size) :
        assert isinstance(size, int) and 3<= size <=50, \
            f"Indiquer des dimensions de labyrinthe cohérentes (entre 3 et 50) : {size}"
        
        self.nb_nodes = size*size   # Nombre de noeuds total
        self.size = size        # Nombre de lignes / colonnes
        self.color = (0,0,0)    # Couleur par défaut pour l'affichage
        self.max_weight = 0     # Poids maximum autorisé pour une arête
        
        self.nodes = []         # Liste des noeuds

        
    # Poids maximum d'une arête
    def set_max_weight(self, weight) :
        self.max_weight = weight
    
    # Couleur du graphe
    def set_color(self, color) :
        self.color = color
   
    
    
    # Ajout d'un noeud adjacent à un noeud
    def add_child(self, parent, num, weight) :
        assert isinstance(weight, int) and 0 < weight < 100, \
            f"Poids négatif ou trop grand (>99) : {weight}"
        self.nodes[parent].append(Node(num, weight))
    
    
    # Initialisation du graphe
    def init_graph(self) :
        # Ajout de chaque noeud en tant que parent
        for i in range(self.nb_nodes) :
            self.nodes.append([Node(i,0)])
        
        # Détermination des labels et arêtes entre chaque noeud
        for i in range(self.size) :
            for j in range(self.size) : 
                num_par = i*self.size + j             # Numéro du futur parent
                num_child_east = num_par + 1          # Numéro d'un éventuel enfant à droite
                num_child_south = num_par + self.size # Même chose en bas
                
            
                # Création d'un enfant à droite si possible
                if j < self.size - 1: # Si on est n'est pas au bord à droite
                    rand = randint(1, self.max_weight) # Valeurs aléatoires des poids
                    self.add_child(num_par, num_child_east, rand)
                    
                    # Graphe non-orienté, ajout du lien en retour (échange parent / enfant)
                    self.add_child(num_child_east, num_par, rand)
                
                # Création d'un enfant en dessous si possible
                if i < self.size - 1: # Si on n'est pas sur la dernière ligne
                    rand = randint(1, self.max_weight) # Valeurs aléatoires des poids
                    self.add_child(num_par, num_child_south, rand)
                    
                    # Graphe non-orienté, ajout du lien en retour (échange parent / enfant)
                    self.add_child(num_child_south, num_par, rand)
            
        
        
    def delete_children(self,num_par,num_chd) :
        # Parcours de tous les noeuds
        for nodes in self.nodes :
            # On ne supprime pas l'enfant qui est relié à son parent "optimal"
            if nodes[0].value == num_par : 
                continue                       
            
            """On parcourt seulement les enfants des autres noeuds 
              (d'où le range(1,len(children))) """
            for i in range(1,len(nodes)) : 
                """ S'il s'agit de celui cherché,
                  on l'efface et une fois supprimé, on quitte la boucle """
                if nodes[i].value == num_chd :  
                    del(nodes[i])          
                    break                       
                                                  
    
    # Indique si tous les parents ont été utilisés (fin de l'algorithme)
    def all_on_use(self) :   
        for nodes in self.nodes :
            if not nodes[0].on_use :  # Noeud non utilisé
                return False
        
        return True
                
        
        
    def prim_algorithm(self, val_dep) :
        self.nodes[val_dep][0].on_use = True     # Noeud de départ
        
        """ Suppression de tous les enfants correspondant au noeud de départ :
          le '-1' en tant que parent permet de parcourir toute la liste par défaut 
          et de le supprimer partout en tant qu'enfant """
        self.delete_children(-1,val_dep)
            
        
        # Tous les parents doivent être traités (condition d'arrêt)
        while not self.all_on_use() :
        
            min_weight = 100        # Stocke le poids minimum (valeur très grande par défaut)
            best_child_num = -1     # Stocke le numéro de l'enfant le plus proche (négatif par défaut)
            best_parent_num = -1    # Stocke le numéro du parent du précédent (négatif par défaut)
                                             
            # Parcours de chaque noeud de la variable self.nodes    
            for nodes in self.nodes :
                # Seuls les parents `on_use` sont traités
                if not nodes[0].on_use :     
                    continue                         
                
                # Parcours des noeuds adjacents de chaque noeud
                for i in range(1,len(nodes)) : 
                    """ Si on trouve un nouveau chemin intéressant et que le noeud adjacent 
                     n'est pas encore utilisé en tant que fils """
                    if nodes[i].weight < min_weight and not nodes[i].is_used :     
                        best_child_num = nodes[i].value   # Nouveau meilleur enfant,
                        best_parent_num = nodes[0].value  # avec son parent
                        min_weight = nodes[i].weight      # Nouveau poids minimum
            
            # Contrôle de la cohérence des résultats obtenus, problème d'initialisation ?        
            assert best_child_num != -1 and best_parent_num != -1, \
                    "Pas d'enfant / parent optimaux ? (indice `-1`)"
            
            # Parcours possible de noeuds adjacents d'un nouveau parent
            self.nodes[best_child_num][0].on_use = True
            
            # Suppression de ce noeud en tant qu'enfant éventuel d'autres parents
            self.delete_children(best_parent_num, best_child_num)  
            
            # Désactivation du chemin qui vient d'être traité
            for node in self.nodes[best_parent_num] :  
                if node.value == best_child_num :
                    node.is_used = True
        
        
        
    # Surcharge de "print"
    # Affiche le graphe / arbre : chaque parent avec ses enfants + valeur, poids    
    def __str__(self) :
        txt = ""
        for nodes in self.nodes :
            txt += f"Parent, Val = {nodes[0].value} // "
            for i in range(1,len(nodes)) :
                txt += f"Val = {nodes[i].value} - Wei = {nodes[i].weight} / " 
            txt += f"\n"
            
        return txt
    

##################################################################################################
##################################################################################################
##################################################################################################

In [ ]:
##################################################################################################
##################################################################################################
##################################################################################################

# Gestion de paramètres de l'écran : taille, couleur de fond
class Screen :
    ################# Question 1 : Création du CONSTRUCTEUR ICI #################
    
    def __init__(self,size) : # La variable `size` est la taille de l'écran
        #######  A COMPLETER  ########
        #######  A COMPLETER  ########
        #######  A COMPLETER  ########  
        self.my_screen = set_mode((size,size)) # Création du contexte d'affichage
        
    #############################################################################    
        
    # Setter   
    # Couleur du fond d'écran
    def set_color(self,r,v,b) :
        pass
        #######  A COMPLETER  ########  
    
    # Chargement de l'image de fond
    def set_image(self,f_name) :
        self.image = pygame.image.load(f_name).convert()
     
    
    # Affichage de l'image (fond écran)
    def blit(self) :
        self.my_screen.blit(self.image,(0,0))


##################################################################################################
##################################################################################################
##################################################################################################

In [ ]:
# Gestion des événements
class Event :
    # Constructeur par défaut
    def __init__(self) :
        return None
    
    # Fermeture de la fenêtre
    def on_close(self) :
        pass
        ################# Question 2 : Gestion de l'arrêt du programme #################
        
        ################### INSTRUCTIONS A COMPLETER ICI ##################
        
        ################################################################################
                


##################################################################################################
##################################################################################################
##################################################################################################

In [ ]:
# Gestion du labyrinthe
class Maze :
    
    def __init__(self,screen,graph) :
        self.graph = graph      # Pointe vers le graphe (créé précédemment)
        self.screen = screen    # Gestion du contexte (créé précédemment)
        self.event = Event()    # Gestion de l'arrêt du programme (créé précédemment)
        
        self.size = 0 # Epaisseur d'un chemin du labyrinthe
        self.step = 0 # Distance entre chaque noeud
    
    
    # Accesseurs
    # Epaisseur / Longueur des chemins du labyrinthe 
    def set_size(self) :  
        # Choix de l'épaisseur des chemins, le `0.6` est empirique
        self.size = 0.6*(self.screen.size/self.graph.size)
        
        """ Pas entre chaque noeud en largeur (et hauteur) en tenant compte
         du fait que le premier node est à self.size et le dernier self.size du bord """
        ####################### INSTRUCTION A COMPLETER ICI ##########################
        
        ##############################################################################
    
    
    # Génération des coordonnées de chaque noeud de la liste de listes
    # `self.graph.nodes`
    def init_maze(self) :
        """
        for nodes in self.graph.nodes :  # Parcours de chaque sous-liste     
                for node in nodes :      # Parcours de chaque noeud
                    
                    ####################### INSTRUCTIONS A COMPLETER ICI ##########################
        
                    
            
                    # Mise à jour des coordonnées de chaque noeud (tuple)
                    
                    
                    ############################################################################## """
     
    # Dessin des noeuds
    def draw_init_maze(self) :
        # Affichage des noeuds
        for i in range(self.graph.size) :
            for j in range(self.graph.size) :
                value = i*self.graph.size + j
                
                # Le coefficient 0.7 est empirique                                   
                circle(self.screen.my_screen, self.graph.color, \
                       self.graph.nodes[value][0].coords,0.7*self.size)
                    
                    
                    
    # Dessin du labyrinthe parfait
    def draw_maze(self) :
        self.draw_init_maze()
           
        # Affichage des liens parents/enfants (rectangles formant le labyrinthe)
        for id_parent in range(self.graph.nb_nodes) :  # `id_parent` est le numéro du noeud parent 
            parent = self.graph.nodes[id_parent][0]    # Le premier élément de chaque sous-liste est le parent
                                                   
            for child in self.graph.nodes[id_parent] :
                id_child = child.value                 # `id_child` est le numéro du noeud enfant
                
                # Le parent est après l'enfant : rect horizontal partant de l'enfant                                   
                if id_parent - id_child == 1 :  
                    rect(self.screen.my_screen, self.graph.color, (child.coords[0]-self.size//2,  \
                            child.coords[1]-self.size//2, self.step + self.size//2, self.size))
                
                ##################################################################################################
                ################### Question 5 : ECRIRE LE CAS PARENT AVANT ENFANT ICI ###########################
                ##################################################################################################
                
                       ####################### INSTRUCTIONS A COMPLETER ICI ##########################      
        
                ##################################################################################################
                
                
                # Le parent est en-dessous de l'enfant : rect vertical partant de l'enfant    
                elif id_parent - id_child == self.graph.size : 
                    rect(self.screen.my_screen, self.graph.color, (child.coords[0]-self.size//2,   \
                            child.coords[1]-self.size//2, self.size, self.step + self.size//2))
                    
                ###################################################################################################
                ################ Question 6 : ECRIRE LE CAS PARENT AU-DESSUS DE L'ENFANT ICI ######################
                ###################################################################################################                 
           
                      ####################### INSTRUCTIONS A COMPLETER ICI ##########################   
        
                 #################################################################################################
                
                # Dans les autres cas, on passe (parent sans enfant)
                else :
                    continue
               
            
            
    # Fin de l'application                
    def on_close(self) :
        self.event.on_close()
        
        
#################################################################################################
#################################################################################################
#################################################################################################

In [ ]:
###########################################################################################
############################## DONNEES CONSTANTES #########################################
###########################################################################################

""" Nombre de colonnes / Lignes du labyrinthe / Node de départ aléatoire (algorithme de Prim)
  Prendre des valeurs COHERENTES entières entre 3 et 50 """
GRAPH_SIZE = 3

# Noeud de départ aléatoire
DEPART_NODE = randint(0,GRAPH_SIZE*GRAPH_SIZE - 1)

# Couleur du labyrinthe. Code couleur Rouge Vert Bleu habituel. Entre 0 et 255 par valeur
GRAPH_COLOR = (5,7,180)

""" Génération au hasard. Valeur entière au moins égale à 2. 
  Remarque : pas de différences sensibles au-delà de 5, il s'agit des poids entre les sommets 
  ATTENTION : ne pas dépasser 99 (inclus)"""
WEIGHT_MAX_RANDOM = 5

""" Taille de l'écran. 
  ATTENTION : Ne pas dépasser ici 1200 (taille du fond d'écran chargé) """
SCREEN_SIZE = 600  

###########################################################################################


In [ ]:
# Initialisation du graphe
graph = Graph(GRAPH_SIZE)                  # Nombre de lignes / colonnes
graph.set_color(GRAPH_COLOR)               # Couleur du labyrinthe
graph.set_max_weight(WEIGHT_MAX_RANDOM)    # Poids maximal d'une arête
graph.init_graph()                         # Initialisation du graphe par défaut

################## AFFICHE LE GRAPHE INITIAL !! ###############################
print(f"Répartition des poids dans le graphe : \n")

###############################################################

# Algorithme de Prim
graph.prim_algorithm(DEPART_NODE)

################## POUR TEST !! ###############################
print(f"\nArbre couvrant de poids minimal :\n ")
print(graph)
###############################################################

In [ ]:
# Lancement de la bibliothèque pygame 
pygame.init()
pygame.display.set_caption("Maze's generation")
      
# Création d'une fenêtre de dessin largeur / hauteur
screen = Screen(SCREEN_SIZE)
                
# Fond d'écran (taille maximale 1200 x 1200 environ pour celle chargée)
screen.set_image("Texture_Mur.jpg")

# Initialisation du labyrinthe
maze = Maze(screen, graph)
maze.set_size()
maze.init_maze()


# Affichage du résultat et gestion du "quit"
while screen.is_running :
    
    # Affichage de l'écran et du labyrinthe
    screen.blit()
    maze.draw_init_maze()  # Affiche seulement les noeuds, écrire maze.draw_maze() pour le labyrinthe
    
    # Affichage effectif
    flip()
    
    # Gestion de la fermeture de la fenêtre (clic sur la croix)
    maze.on_close()
    
    pygame.time.delay(20)  # Une image toutes les 20 ms  


# Fermeture du programme
pygame.display.quit()    # Ferme la fenêtre
pygame.quit()